In [2]:
import sys
import os
import urllib3
from configparser import ConfigParser

# Add your local ThreatConnect SDK to path
sys.path.append("Z:/HTOC/Data_Analytics/threatconnect")
from ThreatConnect import ThreatConnect
from RequestObject import RequestObject
from Owners import Owners

# Add your project repo to path
project_root = r"C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull"
if project_root not in sys.path:
    sys.path.append(project_root)

from utils.config_loader import load_config

# Load API config
config_path = os.path.join(project_root, "utils", "config.json")
try:
    api_secret_key, api_access_id, api_base_url, api_default_org = load_config(config_path)
    print(f"Loaded config from: {config_path}")
    print(f"Base URL: {api_base_url}")
    print(f"Access ID: {api_access_id}")
    print(f"Default Org: {api_default_org}")
except Exception as e:
    print(f"[ERROR] Failed to load configuration: {e}")
    sys.exit(1)

# Disable SSL verification warnings (use cautiously)
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)
verify_ssl = False

# Initialize ThreatConnect session
try:
    tc = ThreatConnect(api_access_id, api_secret_key, api_default_org, api_base_url)
    print("ThreatConnect initialized.")
except Exception as e:
    print(f"[ERROR] Failed to initialize ThreatConnect: {e}")
    sys.exit(1)

# Define the owner (organization scope)
owner = 'HTOC Org'

# Create a request object to fetch indicators (or other data)
try:
    ro = RequestObject()
    ro.set_http_method('GET')
    ro.set_owner(owner)
    ro.set_owner_allowed(True)
    # ro.set_resource_pagination(True)  # Uncomment if needed
    print("RequestObject successfully created.")
except Exception as e:
    print(f"[ERROR] Failed to initialize RequestObject: {e}")
    sys.exit(1)



Loaded config from: C:\Users\jaskew\Documents\project_repository\scripts\Data Movement\ThrearConnect-api-pull\utils\config.json
Base URL: https://hvs.threatconnect.com/api
Access ID: 09783848890162390382
Default Org: HTOC Org
ThreatConnect initialized.
RequestObject successfully created.


In [3]:
import pandas as pd
from datetime import datetime, timedelta
import urllib.parse
import pytz
from collections import defaultdict

# --- ThreatConnect setup (assumes 'tc' and 'ro' objects already exist) ---
# Make sure tc and ro are defined in your environment based on your SDK usage

# Define the time window (3 days ago at 00:00 UTC)
start_date = (datetime.now(pytz.UTC) - timedelta(days=3)).date()
start = f"{start_date}T00:00:00Z"

# Indicator types to query
type_names = [
    "Address", "EmailAddress", "File", "Host", "URL", "ASN", "CIDR",
    "Email Subject", "Hashtag", "Mutex", "Registry Key", "Stripped URL", "User Agent"
]
type_name_condition = ", ".join([f'"{t}"' for t in type_names])

# Owners to pull from
list_of_owners = ['HTOC Org']
final_results = []

for owner in list_of_owners:
    print(f"\nQuerying owner: {owner}")
    try:
        tql_raw = (
            f'ownerName EQ "{owner}" AND '
            f'typeName IN ({type_name_condition}) AND '
            f'lastObserved >= "{start}"'
        )
        tql_encoded = urllib.parse.quote(tql_raw)

        ro.set_http_method('GET')
        ro.set_request_uri(
            f'/v3/indicators?tql={tql_encoded}&fields=tags,observations&resultStart=0&resultLimit=10000'
        )

        response = tc.api_request(ro)

        if response.headers.get('content-type') == 'application/json':
            results = response.json()
            final_results.append(results)
        else:
            print(f"Unexpected response format: {response.headers.get('content-type')}")
    except Exception as e:
        print(f"Failed to query indicators for {owner}: {e}")

# Normalize and clean the indicators
if final_results:
    normalized_data = []
    for result in final_results:
        if 'data' in result:
            for item in result['data']:
                if 'summary' in item:
                    normalized_data.append(item)

    if normalized_data:
        observed_src = pd.json_normalize(normalized_data)
        observed_src['indicator'] = observed_src['summary'].str.split(' ', expand=True)[0].str.strip()
        observed_src = observed_src.drop_duplicates(subset='indicator', keep='first')

        # Ensure lastObserved is a datetime object for filtering
        observed_src['lastObserved'] = pd.to_datetime(observed_src['lastObserved'], utc=True)
        start_dt = pd.to_datetime(start)
        observed_src = observed_src[observed_src['lastObserved'] >= start_dt]

        print(f"\nRetrieved {len(observed_src)} unique observed indicators.")

        # -------------------------------
        # Enrich indicators with VirusTotalV3
        # -------------------------------
        indicator_ids = observed_src['id'].dropna().unique().tolist()
        enriched_results = []

        print(f"\nEnriching {len(indicator_ids)} indicators with VirusTotalV3...")

        for indicator_id in indicator_ids:
            try:
                enrich_url = f'/v3/indicators/{indicator_id}/enrich?type=VirusTotalV3'
                ro.set_http_method('POST')
                ro.set_request_uri(enrich_url)
                ro.set_body({})  # Enrichment call usually has no body

                enrich_response = tc.api_request(ro)

                if enrich_response.status_code == 200:
                    enrich_data = enrich_response.json()
                    enrich_data['indicator_id'] = indicator_id
                    enriched_results.append(enrich_data)

            except Exception as e:
                continue

        # Convert enriched data into DataFrame
        if enriched_results:
            df_enriched = pd.json_normalize(enriched_results)
            df_enriched.rename(columns={'indicator_id': 'id'}, inplace=True)

            # Merge enrichment data into main dataframe
            observed_src = observed_src.merge(df_enriched, on='id', how='left')
            print(f"\nSuccessfully enriched and merged {len(df_enriched)} indicators.")
        else:
            print("\nNo enrichment data retrieved.")

    else:
        print("\nNo valid indicators with 'summary' key retrieved.")
else:
    print("\nNo indicators retrieved.")




Querying owner: HTOC Org

Retrieved 855 unique observed indicators.

Enriching 855 indicators with VirusTotalV3...


Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress kathy@finnaccounting.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress ginny.williams@frcohio.org cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress cameron@yourpensionmeeting.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress no-reply@thryv.com cannot be enriched with VirusTotal because the indicator type isn\'t supported.","status":"Error"}'
Status Code: 400
Failed API Response: b'{"errCode":"0x1001","message":"The EmailAddress k.baker@positiveproximity.com cannot be enriched with Viru


Successfully enriched and merged 832 indicators.


In [8]:
observed_src

,id,dateAdded,ownerId,ownerName,webLink,type,lastModified,rating,confidence,source,...,data.summary,data.privateFlag,data.active,data.activeLocked,data.ip,data.legacyLink,data.enrichment.data,data.hostName,data.dnsActive,data.whoisActive
0,6755399460008466,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T13:42:03Z,1.0,56,Treasury,...,46.235.230.124,False,True,False,46.235.230.124,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 1}]",NaN,NaN,NaN
1,6755399460008449,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T13:42:03Z,1.0,56,Treasury,...,38.45.254.5,False,True,False,38.45.254.5,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 1}]",NaN,NaN,NaN
2,6755399460008438,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T13:42:03Z,1.0,56,Treasury,...,23.224.193.42,False,True,False,23.224.193.42,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 2}]",NaN,NaN,NaN
3,6755399460008419,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T13:42:03Z,1.0,56,Treasury,...,201.254.226.17,False,True,False,201.254.226.17,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 0}]",NaN,NaN,NaN
4,6755399460008413,2025-07-02T12:05:37Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Address,2025-11-03T13:42:03Z,1.0,56,Treasury,...,200.90.52.154,False,True,False,200.90.52.154,https://hvs.threatconnect.com/auth/indicators/...,"[{'type': 'VirusTotal', 'vtMaliciousCount': 1}]",NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
850,5265398,2025-01-23T20:41:54Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-31T16:29:51Z,5.0,91,https://careersinpsychology.org,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
851,4303591,2023-03-03T13:53:09Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-24T23:25:10Z,3.0,72,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
852,4476331,2023-11-16T13:43:29Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2025-01-17T23:24:44Z,4.0,85,http://selligenttier.naylorcampaigns.com,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
853,4442727,2023-10-06T23:26:48Z,9,HTOC Org,https://hvs.threatconnect.com/#/details/indica...,Stripped URL,2024-03-28T23:23:43Z,3.0,83,https://google,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
import os
import pandas as pd
from datetime import datetime, timedelta

# Base file path with placeholder for date
base_path = r"Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d{date}.csv"
#base_path = r"C:\Users\jaskew\Documents\project_repository\data\raw\ObservationDataFiles\htoc_opdiv_obs_d{date}.csv"
date_format = "%Y%m%d"

def get_file_paths(base_path, days=3):
    """ Generate file paths for the last `days` days using list comprehension. """
    today = datetime.utcnow()
    dates_to_pull = [(today - timedelta(days=i)).strftime(date_format) for i in range(days)]
    
    # Construct file paths
    file_paths = [base_path.format(date=dt) for dt in dates_to_pull]
    
    # Filter for existing files
    existing_files = [file_path for file_path in file_paths if os.path.exists(file_path)]
    
    if not existing_files:
        print("No files found for the specified date range.")
    else:
        print(f"Files to be loaded: {existing_files}")
    
    return existing_files

def load_observed_data(file_paths):
    """ Load and concatenate observed data from multiple files. """
    data_frames = []

    for file_path in file_paths:
        try:
            df = pd.read_csv(file_path)
            data_frames.append(df)
        except Exception as e:
            print(f"Error reading file {file_path}: {e}")
    
    # Concatenate data
    if data_frames:
        observed_data_df = pd.concat(data_frames, ignore_index=True)
        print(f"Loaded data from {len(data_frames)} files.")
    else:
        observed_data_df = pd.DataFrame()

    return observed_data_df

# Example Usage:
# Fetch file paths for the last 3 days
file_paths = get_file_paths(base_path, days=5)

# Load observed data
observed_data_df = load_observed_data(file_paths)



C:\Users\jaskew\AppData\Local\Temp\ipykernel_30736\1651825034.py:12: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Files to be loaded: ['Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251103.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251102.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251101.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251031.csv', 'Z:/HTOC/Data_Analytics/Data/OpDiv_Observations/htoc_opdiv_obs_d20251030.csv']
Loaded data from 5 files.


In [10]:
import pandas as pd
from datetime import timedelta
from pandas import Timestamp

# Setup cutoff timestamps
cutoff = Timestamp.utcnow()
date_added_cutoff = cutoff - timedelta(days=30)

# Initialize filtered tags DataFrame
filtered_tags = pd.DataFrame()

# Filter API tags and attach metadata
for _, row in observed_src.iterrows():
    tags_data = row.get('tags.data')

    if isinstance(tags_data, list):
        tags = pd.json_normalize(tags_data)
        tags['name'] = tags['name'].astype(str)
        all_tags_list = tags['name'].tolist()

        api_tags = tags[tags['name'].str.contains('API', case=False, na=False)].copy()
        if not api_tags.empty:
            metadata_cols = [
                'summary', 'observations', 'description', 'type',
                'dateAdded', 'lastModified', 'lastObserved', 'webLink', 'data.enrichment.data'
            ]
            for col in metadata_cols:
                value = row.get(col)
                # If value is a list and not a string, assign [value] * len(api_tags)
                if isinstance(value, list) and not isinstance(value, str):
                    api_tags[col] = [value] * len(api_tags)
                else:
                    api_tags[col] = value
            # Fix: assign all_tags as a list of lists, one per row
            api_tags['all_tags'] = [all_tags_list] * len(api_tags)
            filtered_tags = pd.concat([filtered_tags, api_tags], ignore_index=True)

# Convert date columns
filtered_tags['lastObserved'] = pd.to_datetime(filtered_tags['lastObserved'], errors='coerce')
filtered_tags['dateAdded'] = pd.to_datetime(filtered_tags['dateAdded'], errors='coerce')

# Validate required columns
required_columns = ['indicator', 'OpDiv', 'obs_date']
missing_columns = [col for col in required_columns if col not in observed_data_df.columns]
if missing_columns:
    raise ValueError(f"Missing columns in ProcessedObservedData.csv: {missing_columns}")

# Clean 'name' column and match observations
filtered_tags['cleaned_name'] = filtered_tags['name'].str.replace(r'\s+Splunk API$', '', regex=True)
filtered_tags['observed_date'] = pd.NaT

for index, row in filtered_tags.iterrows():
    match = observed_data_df[
        (observed_data_df['indicator'] == row['summary']) &
        (observed_data_df['OpDiv'] == row['cleaned_name'])
    ]
    if not match.empty:
        filtered_tags.at[index, 'observed_date'] = match['obs_date'].iloc[0]

filtered_tags['observed_date'] = pd.to_datetime(filtered_tags['observed_date'], errors='coerce')
filtered_tags.drop(columns=['cleaned_name'], inplace=True)

# Filter recent tags (last 24 hours)
cutoff_naive = cutoff.tz_localize(None) if cutoff.tzinfo else cutoff
recent_tags = filtered_tags[
    (filtered_tags['lastObserved'] >= cutoff - timedelta(hours=24)) &
    (filtered_tags['observed_date'] >= cutoff_naive - timedelta(days=1))
].copy()

# Extract partner from name and group
recent_tags['partner'] = recent_tags['name'].str.replace(' Splunk API', '', regex=False)

partner_groups = (
    recent_tags.groupby('summary')['partner']
    .agg(['nunique', lambda x: ', '.join(sorted(set(x)))])
    .reset_index()
    .rename(columns={'nunique': 'partner_count', '<lambda_0>': 'partners'})
)

recent_tags = recent_tags.merge(partner_groups, on='summary', how='left')

# Remove duplicate summaries
recent_tags = recent_tags.drop_duplicates(subset='summary', keep='first')

# Unnest 'data.enrichment.data' (list of dicts) into separate columns
if 'data.enrichment.data' in recent_tags.columns:
    enrichment_df = pd.json_normalize(
        recent_tags['data.enrichment.data'].dropna().explode()
    )
    enrichment_df.index = recent_tags['data.enrichment.data'].dropna().explode().index
    enrichment_cols = [col for col in enrichment_df.columns if col not in recent_tags.columns]
    # Join enrichment columns back to recent_tags
    recent_tags = recent_tags.join(enrichment_df[enrichment_cols], how='left')

    # Keep only records with vtMaliciousCount > 10
    recent_tags = recent_tags[recent_tags['vtMaliciousCount'] > 10]
    
# Drop unused columns if present
columns_to_drop = [
    'techniqueId', 'tactics.data', 'tactics.count',
    'platforms.data', 'platforms.count', 'partner', 'name', 'data.enrichment.data'
]
recent_tags.drop(columns=[col for col in columns_to_drop if col in recent_tags.columns], inplace=True, errors='ignore')

# Optional: remove certain tags
# recent_tags = recent_tags[~recent_tags['all_tags'].apply(lambda x: 'I&W' in x or 'htoc_wl' in x if isinstance(x, list) else False)]

recent_tags

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
21,30479,2025-11-02T14:39:01Z,185.220.101[.]182 is registered in Germany und...,185.220.101.182,228072,Address,2021-12-15 13:22:24+00:00,2025-11-03T12:55:25Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Zero Day, HHS Splunk Production API, HRSA Spl...",2025-11-03,1,CMS,13.0
34,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3607075,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
92,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11315675,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0
98,35760,2025-11-03T11:58:26Z,INC9235915,165.154.173.226,576476,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0
119,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15655683,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2554,35057,2025-10-31T22:44:21Z,FBI Email Alert May 14 2025,104.152.52.110,2072,Address,2025-05-14 17:44:46+00:00,2025-10-30T08:34:39Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, VA OIS CSOC CTS Splunk, FDA Sp...",2025-11-03,1,FDA,13.0
2629,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.17,1868796,Address,2025-07-28 17:15:14+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",11.0
2653,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.20,1358768,Address,2025-08-13 15:08:38+00:00,2025-10-30T02:15:15Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0
2771,471298,2025-11-03T10:08:20Z,TASK0891295,64.62.156.16,2453608,Address,2025-06-25 17:45:38+00:00,2025-10-29T23:00:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0


In [11]:
from IPython.display import display

# Get all unique partners (split by comma and flatten)
all_partners = set(
    p.strip()
    for partners in recent_tags['partners'].dropna().unique()
    for p in partners.split(',')
)

# Build buckets: each partner gets all rows where it appears in the partners column
partner_buckets = {
    partner: recent_tags[recent_tags['partners'].str.contains(rf'\b{partner}\b', na=False)]
    for partner in all_partners
}

# Show each partner's dataframe as a table
for partner, df in partner_buckets.items():
    print(f"Partner: {partner} ({len(df)} records)")
    display(df)

Partner: CMS (68 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
21,30479,2025-11-02T14:39:01Z,185.220.101[.]182 is registered in Germany und...,185.220.101.182,228072,Address,2021-12-15 13:22:24+00:00,2025-11-03T12:55:25Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[Zero Day, HHS Splunk Production API, HRSA Spl...",2025-11-03,1,CMS,13.0
34,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3607075,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
92,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11315675,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0
98,35760,2025-11-03T11:58:26Z,INC9235915,165.154.173.226,576476,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0
119,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15655683,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2481,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.39,1686644,Address,2025-08-01 14:25:59+00:00,2025-10-31T04:53:33Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0
2494,471298,2025-11-03T10:08:20Z,INC9263763,193.32.162.145,840397,Address,2025-10-02 17:13:20+00:00,2025-10-31T02:34:38Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, DHA, FDA, OS",14.0
2629,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.17,1868796,Address,2025-07-28 17:15:14+00:00,2025-10-30T07:43:54Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",11.0
2653,471298,2025-11-03T10:08:20Z,RITM0606742,64.62.156.20,1358768,Address,2025-08-13 15:08:38+00:00,2025-10-30T02:15:15Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0


Partner: IHS (38 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
34,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3607075,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
92,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11315675,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0
98,35760,2025-11-03T11:58:26Z,INC9235915,165.154.173.226,576476,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0
119,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15655683,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
126,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.66,16189312,Address,2025-07-28 17:15:57+00:00,2025-11-03T12:20:41Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
164,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.68,15160327,Address,2025-07-28 17:15:37+00:00,2025-11-03T11:53:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
618,471298,2025-11-03T10:08:20Z,TASK0902923,65.49.1.50,1420657,Address,2025-07-28 17:16:02+00:00,2025-11-03T09:00:52Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
690,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.212,15350762,Address,2025-07-28 17:34:02+00:00,2025-11-03T08:59:57Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0
743,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.230,15440207,Address,2025-07-28 17:34:08+00:00,2025-11-03T07:41:55Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0
778,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.89,15350906,Address,2025-07-28 17:16:00+00:00,2025-11-03T07:41:55Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0


Partner: HRSA (50 records)


...

,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
34,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3607075,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
92,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11315675,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0
119,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15655683,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
126,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.66,16189312,Address,2025-07-28 17:15:57+00:00,2025-11-03T12:20:41Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
164,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.68,15160327,Address,2025-07-28 17:15:37+00:00,2025-11-03T11:53:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
178,471298,2025-11-03T10:08:20Z,RITM0606742,65.49.1.92,1519073,Address,2025-07-28 17:34:12+00:00,2025-11-03T11:40:26Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0
618,471298,2025-11-03T10:08:20Z,TASK0902923,65.49.1.50,1420657,Address,2025-07-28 17:16:02+00:00,2025-11-03T09:00:52Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
690,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.212,15350762,Address,2025-07-28 17:34:02+00:00,2025-11-03T08:59:57Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0
727,471298,2025-11-03T10:08:20Z,RITM0589984,170.64.134.89,1011327,Address,2025-06-16 17:42:50+00:00,2025-11-03T07:41:56Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,5,"DHA, FDA, HHS, HRSA, OS",13.0
743,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.230,15440207,Address,2025-07-28 17:34:08+00:00,2025-11-03T07:41:55Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0


Partner: FDA (56 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
34,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3607075,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
92,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11315675,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0
98,35760,2025-11-03T11:58:26Z,INC9235915,165.154.173.226,576476,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0
119,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15655683,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
126,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.66,16189312,Address,2025-07-28 17:15:57+00:00,2025-11-03T12:20:41Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
164,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.68,15160327,Address,2025-07-28 17:15:37+00:00,2025-11-03T11:53:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
178,471298,2025-11-03T10:08:20Z,RITM0606742,65.49.1.92,1519073,Address,2025-07-28 17:34:12+00:00,2025-11-03T11:40:26Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0
618,471298,2025-11-03T10:08:20Z,TASK0902923,65.49.1.50,1420657,Address,2025-07-28 17:16:02+00:00,2025-11-03T09:00:52Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
690,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.212,15350762,Address,2025-07-28 17:34:02+00:00,2025-11-03T08:59:57Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0
719,35057,2025-10-31T22:44:21Z,TASK0912447,192.42.116.178,142695,Address,2025-06-16 17:42:12+00:00,2025-11-03T08:33:39Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[OS Splunk API, FDA Splunk API, CMS Splunk API...",2025-11-03,1,FDA,14.0


Partner: OS (60 records)


,id,lastUsed,description,summary,observations,type,dateAdded,lastModified,lastObserved,webLink,all_tags,observed_date,partner_count,partners,vtMaliciousCount
34,471298,2025-11-03T10:08:20Z,RITM0582833,193.163.125.121,3607075,Address,2025-05-21 13:34:32+00:00,2025-11-03T12:55:04Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, VA OIS CSOC CT...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
92,471298,2025-11-03T10:08:20Z,Executive Summary: \n\n118.193.59[.]10 resolv...,118.193.59.10,11315675,Address,2025-02-12 19:49:28+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, Active Scanning: Scanning IP ...",2025-11-03,6,"CMS, DHA, FDA, HRSA, IHS, OS",12.0
98,35760,2025-11-03T11:58:26Z,INC9235915,165.154.173.226,576476,Address,2025-09-12 14:57:07+00:00,2025-11-03T12:36:17Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,4,"CMS, FDA, IHS, OS",11.0
119,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.194,15655683,Address,2025-07-28 17:34:11+00:00,2025-11-03T12:32:45Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
126,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.66,16189312,Address,2025-07-28 17:15:57+00:00,2025-11-03T12:20:41Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",13.0
164,471298,2025-11-03T10:08:20Z,TASK0902923,198.235.24.68,15160327,Address,2025-07-28 17:15:37+00:00,2025-11-03T11:53:10Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
178,471298,2025-11-03T10:08:20Z,RITM0606742,65.49.1.92,1519073,Address,2025-07-28 17:34:12+00:00,2025-11-03T11:40:26Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,6,"CMS, DHA, FDA, HHS, HRSA, OS",12.0
398,35760,2025-11-03T11:58:26Z,RITM0588707 / TASK0886419,78.153.140.151,84691,Address,2025-06-11 14:46:24+00:00,2025-11-03T11:33:19Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,1,OS,12.0
618,471298,2025-11-03T10:08:20Z,TASK0902923,65.49.1.50,1420657,Address,2025-07-28 17:16:02+00:00,2025-11-03T09:00:52Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",12.0
690,471298,2025-11-03T10:08:20Z,TASK0902923,205.210.31.212,15350762,Address,2025-07-28 17:34:02+00:00,2025-11-03T08:59:57Z,2025-11-03 00:00:00+00:00,https://hvs.threatconnect.com/#/details/indica...,"[DHA Splunk API, OS Splunk API, FDA Splunk API...",2025-11-03,7,"CMS, DHA, FDA, HHS, HRSA, IHS, OS",11.0


import os
import re
import time
import win32com.client as win32

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'

# Start Outlook
outlook = win32.Dispatch("Outlook.Application")
namespace = outlook.GetNamespace("MAPI")
drafts_folder = namespace.GetDefaultFolder(16)  # olFolderDrafts = 16

for partner, df in partner_buckets.items():
    try:
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)
        partner_folder = os.path.join(Tippers_Path, safe_partner)
        os.makedirs(partner_folder, exist_ok=True)

        # Save CSV to disk (Outlook needs a real file to attach)
        csv_filename = f"{safe_partner}_tippers.csv"
        csv_path = os.path.join(partner_folder, csv_filename)
        df.to_csv(csv_path, index=False)
        time.sleep(0.5)  # ensure write completes

        # Create mail item and save to Drafts
        mail = outlook.CreateItem(0)
        mail.Subject = f"Tippers Data for {partner}"
        mail.To = f"{safe_partner.lower()}@yourdomain.com"
        mail.Body = f"Attached is the Tippers CSV file for {partner}."
        mail.Attachments.Add(Source=csv_path)
        
        # Move to Drafts folder
        mail = mail.Move(drafts_folder)
        print(f"Draft saved in Outlook for: {partner}")

    except Exception as e:
        print(f"Error processing partner '{partner}': {e}")


import os
import re
import time
from datetime import datetime
import pandas as pd
import win32com.client as win32

Tippers_Path = r'Z:\HTOC\HTOC Reports\Tippers'
os.makedirs(Tippers_Path, exist_ok=True)

# Add today's date to the filename
today_str = datetime.utcnow().strftime("%Y%m%d")
excel_path = os.path.join(Tippers_Path, f"tippers_by_partner_{today_str}.xlsx")

with pd.ExcelWriter(excel_path, engine='xlsxwriter') as writer:
    for partner, df in partner_buckets.items():
        # Excel sheet names can't be longer than 31 chars or contain special chars
        safe_partner = re.sub(r'[^a-zA-Z0-9_-]', '_', partner)[:31]

        # Convert all timezone-aware datetime columns to naive
        for col in df.select_dtypes(include=['datetimetz']).columns:
            df[col] = df[col].dt.tz_localize(None)

        # Write to Excel first
        df.to_excel(writer, sheet_name=safe_partner, index=False)

        # Then get the worksheet object to format
        worksheet = writer.sheets[safe_partner]
        worksheet.autofilter(0, 0, len(df), len(df.columns) - 1)
        worksheet.freeze_panes(1, 0)

        for i, col in enumerate(df.columns):
            # Set width based on max text length in the column, or column name
            max_len = max(
                df[col].astype(str).map(len).max() if not df.empty else 0,
                len(str(col))
            )
            worksheet.set_column(i, i, min(max_len + 2, 50))  # Cap width at 50

print(f"Excel file with partner tabs saved to: {excel_path}")
